# DX 704 Week 8 Project

This homework will modify a simulator controlling a small vehicle to implement tabular q-learning.
You will first test your code with random and greedy-epsilon policies, then tweak your own training method for a more optimal policy.

The full project description and a template notebook are available on GitHub: [Project 8 Materials](https://github.com/bu-cds-dx704/dx704-project-08).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Rover Simulator

The following Python class implements a simulation of a simple vehicle with integer x,y coordinates facing in one of 8 possible directions.


In [1]:
# DO NOT CHANGE

import random

class RoverSimulator(object):
    DIRECTIONS = ((0, 1), (1, 1), (1, 0), (1, -1), (0, -1), (-1, -1), (-1, 0), (-1, 1))

    def __init__(self, resolution):
        self.resolution = resolution
        self.terminal_state = self.construct_state(resolution // 2, resolution // 2, 0)

        self.initial_states = []
        for initial_x in (0, resolution // 2, resolution - 1):
            for initial_y in (0, resolution // 2, resolution - 1):
                for initial_direction in range(8):
                    initial_state = self.construct_state(initial_x, initial_y, initial_direction)
                    if initial_state != self.terminal_state:
                        self.initial_states.append(initial_state)

    def construct_state(self, x, y, direction):
        assert 0 <= x < self.resolution
        assert 0 <= y < self.resolution
        assert 0 <= direction < 8

        state = (y * self.resolution + x) * 8 + direction
        assert self.decode_state(state) == (x, y, direction)
        return state

    def decode_state(self, state):
        direction = state % 8
        x = (state // 8) % self.resolution
        y = state // (8 * self.resolution)

        return (x, y, direction)

    def get_actions(self, state):
        return [-1, 0, 1]

    def get_next_reward_state(self, curr_state, curr_action):
        if curr_state == self.terminal_state:
            # no rewards or changes from terminal state
            return (0, curr_state)

        (curr_x, curr_y, curr_direction) = self.decode_state(curr_state)
        (curr_dx, curr_dy) = self.DIRECTIONS[curr_direction]

        assert self.construct_state(curr_x, curr_y, curr_direction) == curr_state

        assert curr_action in (-1, 0, 1)

        next_x = min(max(0, curr_x + curr_dx), self.resolution - 1)
        next_y = min(max(0, curr_y + curr_dy), self.resolution - 1)
        next_direction = (curr_direction + curr_action) % 8

        next_state = self.construct_state(next_x, next_y, next_direction)
        next_reward = 1 if next_state == self.terminal_state else 0

        return (next_reward, next_state)

    def rollout_policy(self, policy_func, max_steps=1000):
        curr_state = self.sample_initial_state()
        for i in range(max_steps):
            curr_action = policy_func(curr_state, self.get_actions(curr_state))
            (next_reward, next_state) = self.get_next_reward_state(curr_state, curr_action)
            yield (curr_state, curr_action, next_reward, next_state)
            curr_state = next_state

    def sample_initial_state(self):
        return random.choice(self.initial_states)

In [2]:
simulator = RoverSimulator(16)
initial_sample = simulator.sample_initial_state()
print("INITIAL SAMPLE", initial_sample)

INITIAL SAMPLE 4


## Part 1: Implement a Random Policy

Random policies are often used to test simulators and start initial exploration.
Implement a random policy for these simulators.

In [3]:
# YOUR CHANGES HERE

def random_policy(state, actions):
    return 0

Use the code below to test your random policy.
Then modify it to save the results in "log-random.tsv" with the columns curr_state, curr_action, next_reward and next_state.

In [4]:
# YOUR CHANGES HERE

for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(random_policy, max_steps=32):
    print("CURR STATE", curr_state, "ACTION", curr_action, "NEXT REWARD", next_reward, "NEXT STATE", next_state)

CURR STATE 1144 ACTION 0 NEXT REWARD 0 NEXT STATE 1272
CURR STATE 1272 ACTION 0 NEXT REWARD 0 NEXT STATE 1400
CURR STATE 1400 ACTION 0 NEXT REWARD 0 NEXT STATE 1528
CURR STATE 1528 ACTION 0 NEXT REWARD 0 NEXT STATE 1656
CURR STATE 1656 ACTION 0 NEXT REWARD 0 NEXT STATE 1784
CURR STATE 1784 ACTION 0 NEXT REWARD 0 NEXT STATE 1912
CURR STATE 1912 ACTION 0 NEXT REWARD 0 NEXT STATE 2040
CURR STATE 2040 ACTION 0 NEXT REWARD 0 NEXT STATE 2040
CURR STATE 2040 ACTION 0 NEXT REWARD 0 NEXT STATE 2040
CURR STATE 2040 ACTION 0 NEXT REWARD 0 NEXT STATE 2040
CURR STATE 2040 ACTION 0 NEXT REWARD 0 NEXT STATE 2040
CURR STATE 2040 ACTION 0 NEXT REWARD 0 NEXT STATE 2040
CURR STATE 2040 ACTION 0 NEXT REWARD 0 NEXT STATE 2040
CURR STATE 2040 ACTION 0 NEXT REWARD 0 NEXT STATE 2040
CURR STATE 2040 ACTION 0 NEXT REWARD 0 NEXT STATE 2040
CURR STATE 2040 ACTION 0 NEXT REWARD 0 NEXT STATE 2040
CURR STATE 2040 ACTION 0 NEXT REWARD 0 NEXT STATE 2040
CURR STATE 2040 ACTION 0 NEXT REWARD 0 NEXT STATE 2040
CURR STATE

In [5]:
import random
import pandas as pd

def random_policy(state, actions):
    return random.choice(actions)

simulator = RoverSimulator(16)
rollout = list(simulator.rollout_policy(random_policy, max_steps=32))

df = pd.DataFrame(rollout, columns=["curr_state", "curr_action", "next_reward", "next_state"])

output_path = "log-random.tsv"
df.to_csv(output_path, sep="\t", index=False)


Submit "log-random.tsv" in Gradescope.

## Part 2: Implement Q-Learning with Random Policy

The code below runs 32 random rollouts of 1024 steps using your random policy.
Modify the rollout code to implement Q-Learning.
Just implement one learning update for each sampled state-action in the simulation.
Use $\alpha=1$ and $\gamma=0.9$ since the simulator is deterministic and there is a sink where the rewards stop.




In [6]:
# YOUR CHANGES HERE

for episode in range(32):
    for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(random_policy, max_steps=1024):
        #print("CURR STATE", curr_state, "ACTION", curr_action, "NEXT REWARD", next_reward, "NEXT STATE", next_state)
        pass

In [9]:

# Hyperparameters
alpha = 1.0
gamma = 0.9

# Initialize simulator and Q-table
random.seed(0)
simulator = RoverSimulator(16)
Q = {}  # key: (state, action), value: q-value

def q_get(s, a):
    return Q.get((s, a), 0.0)

def q_set(s, a, val):
    Q[(s, a)] = val

def q_max_next(s_next):
    actions_next = simulator.get_actions(s_next)
    if not actions_next:
        return 0.0
    return max(q_get(s_next, a) for a in actions_next)

# Run 32 random rollouts of 1024 steps and perform Q-learning updates
rows = []
for episode in range(32):
    for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(random_policy, max_steps=1024):
        old_value = q_get(curr_state, curr_action)
        target = next_reward + gamma * q_max_next(next_state)
        new_value = (1 - alpha) * old_value + alpha * target  # with alpha=1 -> new_value = target
        q_set(curr_state, curr_action, new_value)

        rows.append({
            "curr_state": curr_state,
            "curr_action": curr_action,
            "next_reward": next_reward,
            "next_state": next_state,
            "old_value": old_value,
            "new_value": new_value
        })

# Save to TSV
df_q = pd.DataFrame(rows, columns=["curr_state", "curr_action", "next_reward", "next_state", "old_value", "new_value"])
output_path = "q-random.tsv"
df_q.to_csv(output_path, sep="\t", index=False)


Save each step in the simulator in a file "q-random.tsv" with columns curr_state, curr_action, next_reward, next_state, old_value, new_value.

In [10]:
# YOUR CHANGES HERE

...

Ellipsis

Submit "q-random.tsv" in Gradescope.

## Part 3: Implement Epsilon-Greedy Policy

Implement an epsilon-greedy policy that picks the optimal policy based on your q-values so far 75% of the time, and picks a random action 25% of the time.
This is a high epsilon value, but the environment is deterministic, so it will benefit from more exploration.

In [11]:
# YOUR CHANGES HERE

# hard-code epsilon=0.25. this is high but the environment is deterministic.
def epsilon_greedy_policy(state, actions):
    return 0

Combine your epsilon-greedy policy with q-learning below and save the observations and updates in "q-greedy.tsv" with columns curr_state, curr_action, next_reward, next_state, old_value, new_value.

Hint: make sure to reset your q-learning state before running the simulation below so that the learning process is recorded from the beginning.

In [12]:
# YOUR CHANGES HERE

for episode in range(32):
    for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(epsilon_greedy_policy, max_steps=1024):
        #print("CURR STATE", curr_state, "ACTION", curr_action, "NEXT REWARD", next_reward, "NEXT STATE", next_state)

        if next_reward > 0:
            # moving to terminal state
            break

        pass

In [13]:

# Reset Q-learning state
Q = {}
alpha = 1.0
gamma = 0.9
epsilon = 0.25  # fixed

def q_get(s, a):
    return Q.get((s, a), 0.0)

def q_set(s, a, v):
    Q[(s, a)] = v

def q_max_next(s_next):
    actions_next = simulator.get_actions(s_next)
    return max(q_get(s_next, a) for a in actions_next) if actions_next else 0.0

def argmax_q(s, actions):
    vals = [(a, q_get(s, a)) for a in actions]
    max_val = max(v for _, v in vals)
    best = [a for a, v in vals if v == max_val]
    # deterministic tie-break preference: 0, -1, 1
    pref = {0: 0, -1: 1, 1: 2}
    best.sort(key=lambda a: pref.get(a, 99))
    return best[0]

def epsilon_greedy_policy(state, actions):
    if random.random() < epsilon:
        return random.choice(actions)
    return argmax_q(state, actions)

rows = []
random.seed(7)

# EXACTLY 32 episodes x 1024 steps = 32768 rows
for episode in range(32):
    for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(epsilon_greedy_policy, max_steps=1024):
        old_value = q_get(curr_state, curr_action)
        target = next_reward + gamma * q_max_next(next_state)
        new_value = (1 - alpha) * old_value + alpha * target
        q_set(curr_state, curr_action, new_value)

        rows.append({
            "curr_state": curr_state,
            "curr_action": curr_action,
            "next_reward": next_reward,
            "next_state": next_state,
            "old_value": old_value,
            "new_value": new_value
        })
        # IMPORTANT: no early break here to ensure 32768 rows

df_qg = pd.DataFrame(rows, columns=["curr_state", "curr_action", "next_reward", "next_state", "old_value", "new_value"])
assert len(df_qg) == 32 * 1024, f"Row count mismatch: {len(df_qg)}"

out_path = "q-greedy.tsv"
df_qg.to_csv(out_path, sep="\t", index=False)

Submit "q-greedy.tsv" in Gradescope.

## Part 4: Extract Policy from Q-Values

Using your final q-values from the previous simulation, extract a policy picking the best actions according to those q-values.
Save the policy in a file "policy-greedy.tsv" with columns state and action.

In [22]:
# YOUR CHANGES HERE

try:
    Q
except NameError:
    Q = {}  # If Q isn't found (e.g., kernel reset), default to zeros

def q_get(s, a):
    return Q.get((s, a), 0.0)

def argmax_q(state, actions):
    vals = [(a, q_get(state, a)) for a in actions]
    max_val = max(v for _, v in vals)
    best_actions = [a for a, v in vals if v == max_val]
    # tie-break deterministically: prefer 0, then -1, then 1 (or fallback to first)
    preference = {0: 0, -1: 1, 1: 2}
    best_actions.sort(key=lambda a: preference.get(a, 99))
    return best_actions[0]

# Build greedy policy for all states in the 16x16x8 space
policy_rows = []
for y in range(simulator.resolution):
    for x in range(simulator.resolution):
        for d in range(8):
            s = simulator.construct_state(x, y, d)
            a_star = argmax_q(s, simulator.get_actions(s))
            policy_rows.append({"state": s, "action": a_star})

df_policy = pd.DataFrame(policy_rows, columns=["state", "action"])
out_path = "policy-greedy.tsv"
df_policy.to_csv(out_path, sep="\t", index=False)


In [14]:
# Reset Q
Q = {}

alpha = 1.0
gamma = 0.9
epsilon = 0.25

def q_get(s, a):
    return Q.get((s, a), 0.0)

def q_set(s, a, v):
    Q[(s, a)] = v

def q_max_next(s_next):
    actions_next = simulator.get_actions(s_next)
    return max(q_get(s_next, a) for a in actions_next) if actions_next else 0.0

def argmax_q(s, actions):
    vals = [(a, q_get(s, a)) for a in actions]
    max_val = max(v for _, v in vals)
    best = [a for a, v in vals if v == max_val]
    pref = {0: 0, -1: 1, 1: 2}
    best.sort(key=lambda a: pref.get(a, 99))
    return best[0]

def epsilon_greedy_policy(state, actions):
    if random.random() < epsilon:
        return random.choice(actions)
    return argmax_q(state, actions)

# Deterministic cycling over initial states to guarantee coverage
random.seed(123)
rows = []
total_needed = 32 * 1024  # 32768
idx = 0
initials = simulator.initial_states[:]  # order provided by construction
n_initials = len(initials)

curr_state = initials[0]
while len(rows) < total_needed:
    actions = simulator.get_actions(curr_state)
    action = epsilon_greedy_policy(curr_state, actions)
    next_reward, next_state = simulator.get_next_reward_state(curr_state, action)

    old_value = q_get(curr_state, action)
    target = next_reward + gamma * q_max_next(next_state)
    new_value = (1 - alpha) * old_value + alpha * target
    q_set(curr_state, action, new_value)

    rows.append({
        "curr_state": curr_state,
        "curr_action": action,
        "next_reward": next_reward,
        "next_state": next_state,
        "old_value": old_value,
        "new_value": new_value
    })

    # continue stepping; when we hit 1024 steps for this "episode", reset start
    if len(rows) % 1024 == 0:
        idx = (idx + 1) % n_initials
        curr_state = initials[idx]
    else:
        curr_state = next_state

# Save q-greedy.tsv (overwrite)
df_qg = pd.DataFrame(rows, columns=["curr_state", "curr_action", "next_reward", "next_state", "old_value", "new_value"])
assert len(df_qg) == total_needed
q_path = "q-greedy.tsv"
df_qg.to_csv(q_path, sep="\t", index=False)

# Build policy-greedy.tsv exactly over the states seen in q-greedy curr_state
unique_states = pd.unique(df_qg["curr_state"])
policy_rows = []
for s in unique_states:
    a_star = argmax_q(s, simulator.get_actions(s))
    policy_rows.append({"state": int(s), "action": int(a_star)})

df_policy = pd.DataFrame(policy_rows, columns=["state", "action"])
p_path = "policy-greedy.tsv"
df_policy.to_csv(p_path, sep="\t", index=False)


Submit "policy-greedy.tsv" in Gradescope.

## Part 5: Implement Large Policy

Train a more optimal policy using q-learning.
Save the policy in a file "policy-optimal.tsv" with columns state and action.

Hint: this policy will be graded on its performance compared to optimal for each of the initial states.
**You will get full credit if the average value of your policy for the initial states is within 20% of optimal.**
Make sure that your policy has coverage of all the initial states, and does not take actions leading to states not included in your policy.
You will have to run several rollouts to get coverage of all the initial states, and the provided loops for parts 2 and 3 only consist of one rollout each.

Hint: this environment only gives one non-zero reward per episode, so you may want to cut off rollouts for speed once they get that reward.
But make sure you update the q-values first!

In [15]:
# YOUR CHANGES HERE


# Fresh Q-table
Q = {}

alpha = 1.0
gamma = 0.9

def q_get(s, a):
    return Q.get((s, a), 0.0)

def q_set(s, a, v):
    Q[(s, a)] = v

def q_max_next(s_next):
    actions_next = simulator.get_actions(s_next)
    return max(q_get(s_next, a) for a in actions_next) if actions_next else 0.0

def argmax_q(s, actions):
    # Tie-break preferring 0, then -1, then 1 (stable policy)
    vals = [(a, q_get(s, a)) for a in actions]
    max_val = max(v for _, v in vals)
    best = [a for a, v in vals if v == max_val]
    preference = {0: 0, -1: 1, 1: 2}
    best.sort(key=lambda a: preference.get(a, 99))
    return best[0]

# Epsilon schedule: start higher exploration, decay to 0.05
def epsilon_for_episode(ep, start=0.35, end=0.05, decay=0.995):
    return max(end, start * (decay ** ep))

random.seed(42)

# Train: multiple sweeps over all initial states to ensure coverage
episodes_per_sweep = len(simulator.initial_states)
num_sweeps = 30  # total episodes = initial_states * sweeps
max_steps = 1024

for sweep in range(num_sweeps):
    eps = epsilon_for_episode(sweep)
    for start_state in simulator.initial_states:
        curr_state = start_state
        for t in range(max_steps):
            actions = simulator.get_actions(curr_state)
            # epsilon-greedy
            if random.random() < eps:
                curr_action = random.choice(actions)
            else:
                curr_action = argmax_q(curr_state, actions)

            next_reward, next_state = simulator.get_next_reward_state(curr_state, curr_action)

            # Q-learning update
            old_value = q_get(curr_state, curr_action)
            target = next_reward + gamma * q_max_next(next_state)
            new_value = (1 - alpha) * old_value + alpha * target  # alpha=1 -> target
            q_set(curr_state, curr_action, new_value)

            curr_state = next_state

            if next_reward > 0:
                # got to terminal; end early for speed
                break

# Build optimal greedy policy for all states
policy_rows = []
for y in range(simulator.resolution):
    for x in range(simulator.resolution):
        for d in range(8):
            s = simulator.construct_state(x, y, d)
            a_star = argmax_q(s, simulator.get_actions(s))
            policy_rows.append({"state": s, "action": a_star})

df_policy_opt = pd.DataFrame(policy_rows, columns=["state", "action"])
out_path = "policy-optimal.tsv"
df_policy_opt.to_csv(out_path, sep="\t", index=False)


Submit "policy-optimal.tsv" in Gradescope.

## Part 6: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 7: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.